In [ ]:


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
def set_seed(seed=42):

    random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset"
TRAIN_IMAGE_PATH = os.path.join(DATASET_PATH, "train", "images")

print(TRAIN_IMAGE_PATH)

In [ ]:
image_files = os.listdir(TRAIN_IMAGE_PATH)

print("Total Images:", len(image_files))
print(image_files[:5])

In [ ]:
simclr_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        size=128,
        scale=(0.7, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5,
        hue=0.1
    ),

    transforms.RandomGrayscale(p=0.2),

    transforms.GaussianBlur(
        kernel_size=5,
        sigma=(0.1, 2.0)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class SimCLRTransform:

    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):

        view1 = self.base_transform(x)
        view2 = self.base_transform(x)

        return view1, view2

In [ ]:
class ObstacleDataset(Dataset):

    def __init__(self, image_dir, transform=None):

        self.image_dir = image_dir
        self.transform = transform

        valid_extensions = (".jpg", ".jpeg", ".png")

        self.image_paths = [
            os.path.join(image_dir, img)
            for img in os.listdir(image_dir)
            if img.lower().endswith(valid_extensions)
        ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

In [ ]:
dataset = ObstacleDataset(
    image_dir=TRAIN_IMAGE_PATH,
    transform=SimCLRTransform(simclr_transform)
)

print("Dataset Size:", len(dataset))

In [ ]:
view1, view2 = dataset[0]

print("View1 Shape:", view1.shape)
print("View2 Shape:", view2.shape)

In [ ]:
def denormalize(img_tensor):

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

    img_tensor = img_tensor * std + mean

    return img_tensor.clamp(0, 1)

In [ ]:
sample1, sample2 = dataset[random.randint(0, len(dataset)-1)]

sample1 = denormalize(sample1)
sample2 = denormalize(sample2)

fig, ax = plt.subplots(1, 2, figsize=(10,5))

ax[0].imshow(sample1.permute(1,2,0))
ax[0].set_title("Augmented View 1")
ax[0].axis("off")

ax[1].imshow(sample2.permute(1,2,0))
ax[1].set_title("Augmented View 2")
ax[1].axis("off")

plt.show()

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

In [ ]:
view1_batch, view2_batch = next(iter(train_loader))

print("View1 Batch Shape:", view1_batch.shape)
print("View2 Batch Shape:", view2_batch.shape)

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(12,6))

for i in range(4):

    img1 = denormalize(view1_batch[i]).permute(1,2,0)
    img2 = denormalize(view2_batch[i]).permute(1,2,0)

    ax[0, i].imshow(img1)
    ax[0, i].set_title("View 1")
    ax[0, i].axis("off")

    ax[1, i].imshow(img2)
    ax[1, i].set_title("View 2")
    ax[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models

In [ ]:
class ResNetSimCLR(nn.Module):

    def __init__(self, projection_dim=128):

        super().__init__()

        # Load ResNet18
        self.encoder = models.resnet18(weights=None)

        # Remove classification layer
        num_features = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()

        # Projection Head
        self.projector = nn.Sequential(

            nn.Linear(num_features, 512),
            nn.ReLU(),

            nn.Linear(512, projection_dim)
        )

    def forward(self, x):

        h = self.encoder(x)

        z = self.projector(h)

        return h, z

In [ ]:
model = ResNetSimCLR(
    projection_dim=128
).to(device)

print(model)

In [ ]:
class NTXentLoss(nn.Module):

    def __init__(self, temperature=0.5):

        super().__init__()

        self.temperature = temperature

    def forward(self, z_i, z_j):

        batch_size = z_i.size(0)

        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)

        representations = torch.cat([z_i, z_j], dim=0)

        similarity_matrix = F.cosine_similarity(
            representations.unsqueeze(1),
            representations.unsqueeze(0),
            dim=2
        )

        # Remove self similarity
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(device)

        similarity_matrix = similarity_matrix.masked_fill(mask, -1e9)

        # Positive pairs
        positives = torch.cat([
            torch.diag(similarity_matrix, batch_size),
            torch.diag(similarity_matrix, -batch_size)
        ], dim=0)

        # Denominator
        logits = similarity_matrix / self.temperature

        exp_logits = torch.exp(logits)

        denominator = exp_logits.sum(dim=1)

        positives_exp = torch.exp(positives / self.temperature)

        loss = -torch.log(positives_exp / denominator)

        return loss.mean()

In [ ]:
criterion = NTXentLoss(
    temperature=0.5
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
def train_simclr(model, loader, optimizer, criterion, epochs):

    

    loss_history = []

    for epoch in range(epochs):
        model.train()

        total_loss = 0

        for view1, view2 in loader:

            view1 = view1.to(device)
            view2 = view2.to(device)

            _, z1 = model(view1)
            _, z2 = model(view2)

            loss = criterion(z1, z2)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)

        loss_history.append(avg_loss)

        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:

            torch.save(
                model.state_dict(),
                f"simclr_epoch_{epoch+1}.pth"
            )

            print(f"Checkpoint Saved at Epoch {epoch+1}")

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

    return loss_history

In [ ]:
EPOCHS = 10

loss_history = train_simclr(
    model=model,
    loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS
)

# Save full SimCLR model
torch.save(
    model.state_dict(),
    "simclr_resnet18.pth"
)

# Save encoder only
torch.save(
    model.encoder.state_dict(),
    "simclr_encoder.pth"
)

print("Model and Encoder Saved")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(loss_history)

plt.title("SimCLR Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.grid(True)

plt.show()

# Masked AutoEncoder

In [ ]:
!pip install timm -q

In [ ]:
import timm

model = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=False
)

print("timm working")

In [ ]:
import timm

import torch.nn as nn
import torch.optim as optim

In [ ]:
mae_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
mae_dataset = ObstacleDataset(
    image_dir=TRAIN_IMAGE_PATH,
    transform=mae_transform
)

print("MAE Dataset Size:", len(mae_dataset))

In [ ]:
mae_loader = DataLoader(
    mae_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

In [ ]:
images = next(iter(mae_loader))

print(images.shape)

In [ ]:
def random_mask(images, mask_ratio=0.75):

    batch_size, channels, height, width = images.shape

    patch_size = 16

    num_patches_h = height // patch_size
    num_patches_w = width // patch_size

    total_patches = num_patches_h * num_patches_w

    num_masked = int(mask_ratio * total_patches)

    masked_images = images.clone()

    for img in masked_images:

        mask_indices = random.sample(
            range(total_patches),
            num_masked
        )

        for idx in mask_indices:

            row = idx // num_patches_w
            col = idx % num_patches_w

            h_start = row * patch_size
            w_start = col * patch_size

            img[
                :,
                h_start:h_start+patch_size,
                w_start:w_start+patch_size
            ] = 0

    return masked_images

In [ ]:
sample_batch = next(iter(mae_loader))

masked_batch = random_mask(sample_batch)

original = denormalize(sample_batch[0]).permute(1,2,0)
masked = denormalize(masked_batch[0]).permute(1,2,0)

fig, ax = plt.subplots(1,2, figsize=(10,5))

ax[0].imshow(original)
ax[0].set_title("Original")
ax[0].axis("off")

ax[1].imshow(masked)
ax[1].set_title("Masked")
ax[1].axis("off")

plt.show()

In [ ]:
class SimpleMAE(nn.Module):

    def __init__(self):

        super().__init__()

        # ViT encoder
        self.encoder = timm.create_model(
            "vit_tiny_patch16_224",
            pretrained=False,
            num_classes=0
        )

        encoder_dim = self.encoder.num_features

        # Decoder
        self.decoder = nn.Sequential(

            nn.Linear(encoder_dim, 1024),
            nn.ReLU(),

            nn.Linear(1024, 3 * 224 * 224)
        )

    def forward(self, x):

        latent = self.encoder(x)

        reconstruction = self.decoder(latent)

        reconstruction = reconstruction.view(
            -1,
            3,
            224,
            224
        )

        return latent, reconstruction

In [ ]:
mae_model = SimpleMAE().to(device)

print(mae_model)

In [ ]:
mae_criterion = nn.MSELoss()

mae_optimizer = optim.Adam(
    mae_model.parameters(),
    lr=1e-4
)

In [ ]:
def train_mae(model, loader, optimizer, criterion, epochs):

    loss_history = []

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for batch_idx, images in enumerate(loader):

            images = images.to(device)

            masked_images = random_mask(
                images,
                mask_ratio=0.75
            ).to(device)

            latent, reconstructed = model(masked_images)

            loss = criterion(reconstructed, images)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 50 == 0:

                print(
                    f"Epoch [{epoch+1}/{epochs}] "
                    f"Batch [{batch_idx}/{len(loader)}] "
                    f"Loss: {loss.item():.4f}"
                )

        avg_loss = total_loss / len(loader)

        loss_history.append(avg_loss)

        print(f"\nEpoch [{epoch+1}/{epochs}] Average Loss: {avg_loss:.4f}\n")

    return loss_history

In [ ]:
MAE_EPOCHS = 10

mae_loss_history = train_mae(
    model=mae_model,
    loader=mae_loader,
    optimizer=mae_optimizer,
    criterion=mae_criterion,
    epochs=MAE_EPOCHS
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(mae_loss_history)

plt.title("MAE Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction Loss")

plt.grid(True)

plt.show()

In [ ]:
torch.save(
    mae_model.state_dict(),
    "simple_mae.pth"
)

torch.save(
    mae_model.encoder.state_dict(),
    "mae_encoder.pth"
)

print("MAE Model and Encoder Saved")

# Analysis

In [91]:
simclr_model = model

In [92]:
analysis_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [93]:
class AnalysisDataset(Dataset):

    def __init__(self, image_dir, transform=None):

        self.transform = transform

        valid_extensions = (".jpg", ".jpeg", ".png")

        self.image_paths = [
            os.path.join(image_dir, img)
            for img in os.listdir(image_dir)
            if img.lower().endswith(valid_extensions)
        ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, img_path

In [94]:
analysis_dataset = AnalysisDataset(
    TRAIN_IMAGE_PATH,
    transform=analysis_transform
)

analysis_loader = DataLoader(
    analysis_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

In [98]:
simclr_model = ResNetSimCLR(
    projection_dim=128
).to(device)

simclr_model.load_state_dict(
    torch.load("simclr_resnet18.pth")
)

simclr_model.eval()

print("SimCLR model loaded")

SimCLR model loaded


In [99]:
simclr_embeddings = []
simclr_paths = []

simclr_model = simclr_model.to(device)

simclr_model.eval()

with torch.no_grad():

    for images, paths in analysis_loader:

        images = images.to(device)

        h, z = simclr_model(images)

        simclr_embeddings.append(h.cpu())

        simclr_paths.extend(paths)

In [100]:
simclr_embeddings = torch.cat(
    simclr_embeddings,
    dim=0
)

print(simclr_embeddings.shape)

torch.Size([19186, 512])


In [101]:
mae_model = SimpleMAE().to(device)

mae_model.load_state_dict(
    torch.load("simple_mae.pth")
)

mae_model.eval()

print("MAE model loaded")

MAE model loaded


In [102]:
mae_embeddings = []

with torch.no_grad():

    for images, _ in analysis_loader:

        images = images.to(device)

        latent, reconstruction = mae_model(images)

        mae_embeddings.append(latent.cpu())

In [103]:
mae_embeddings = torch.cat(
    mae_embeddings,
    dim=0
)

print(mae_embeddings.shape)

torch.Size([19186, 192])


In [104]:
torch.save(
    simclr_embeddings,
    "simclr_embeddings.pt"
)

print("SimCLR embeddings saved")

torch.save(
    mae_embeddings,
    "mae_embeddings.pt"
)

print("MAE embeddings saved")



SimCLR embeddings saved
MAE embeddings saved
